In [7]:
import pandas as pd
import re
import numpy as np
import torch
import random
from itables import show
import pubchempy as pcp
import rdkit
from rdkit import Chem
from rdkit.Chem import Descriptors
from descriptastorus.descriptors import rdDescriptors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from functools import lru_cache
import xgboost
from sklearn.model_selection import train_test_split

In [5]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [6]:
df = pd.read_csv('Cleaned_VLE_Data_with_Smiles_and_Mols_and_Descriptors.csv')
print(df.shape)
df.head()

(106383, 413)


,Unnamed: 0,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,...,"('fr_sulfonamd', <class 'numpy.float64'>).1","('fr_sulfone', <class 'numpy.float64'>).1","('fr_term_acetylene', <class 'numpy.float64'>).1","('fr_tetrazole', <class 'numpy.float64'>).1","('fr_thiazole', <class 'numpy.float64'>).1","('fr_thiocyan', <class 'numpy.float64'>).1","('fr_thiophene', <class 'numpy.float64'>).1","('fr_unbrch_alkane', <class 'numpy.float64'>).1","('fr_urea', <class 'numpy.float64'>).1","('qed', <class 'numpy.float64'>).1"
0,0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
1,1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
2,2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
3,3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493
4,4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,...,1.593061e-17,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493


In [ ]:
# --- Implementation: Create the 'strat_key' ---
# We convert each component to a string and join them with a separator.
# Using a separator like '_' is good practice to prevent ambiguity (e.g., 'A1' + '2' vs. 'A' + '12').
df['strat_key'] = Set(df['Component 1'].astype(str) + '_' + df['Component 2'].astype(str))

print("DataFrame with the new 'strat_key' column:")
df.head()

DataFrame with the new 'strat_key' column:


,Unnamed: 0,property,value,phase,"Temperature, K","Pressure, kPa",Mole fraction,Component 1,Component 2,Smiles 1,...,"('fr_sulfone', <class 'numpy.float64'>).1","('fr_term_acetylene', <class 'numpy.float64'>).1","('fr_tetrazole', <class 'numpy.float64'>).1","('fr_thiazole', <class 'numpy.float64'>).1","('fr_thiocyan', <class 'numpy.float64'>).1","('fr_thiophene', <class 'numpy.float64'>).1","('fr_unbrch_alkane', <class 'numpy.float64'>).1","('fr_urea', <class 'numpy.float64'>).1","('qed', <class 'numpy.float64'>).1",strat_key
0,0,"Thermal conductivity, W/m/K",0.02096,Gas,300.0,724.0,0.2493,carbon dioxide,methane,C(=O)=O,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,carbon dioxide_methane
1,1,"Thermal conductivity, W/m/K",0.02125,Gas,300.0,1288.0,0.2493,carbon dioxide,methane,C(=O)=O,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,carbon dioxide_methane
2,2,"Thermal conductivity, W/m/K",0.02161,Gas,300.0,1835.0,0.2493,carbon dioxide,methane,C(=O)=O,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,carbon dioxide_methane
3,3,"Thermal conductivity, W/m/K",0.02197,Gas,300.0,2335.0,0.2493,carbon dioxide,methane,C(=O)=O,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,carbon dioxide_methane
4,4,"Thermal conductivity, W/m/K",0.02245,Gas,300.0,2820.0,0.2493,carbon dioxide,methane,C(=O)=O,...,5.766101e-14,2.957989e-11,0.168378,0.16738,1.481515e-18,2.324150e-16,4.703598e-08,0.166633,0.187493,carbon dioxide_methane


In [ ]:
# Use .value_counts() to get the raw count of each unique key
combination_counts = df['strat_key'].value_counts()

print("\n--- Absolute Counts of Each Combination ---")
print(combination_counts)
print(f"\nMean count per combination: {combination_counts.mean()}")
print(f"\nMean count per combination: {combination_counts.median()}")



--- Absolute Counts of Each Combination ---
strat_key
ethanol_heptane                                                         1914
methanol_water                                                          1849
acetone_heptane                                                         1747
dimethyl carbonate_hexane                                               1440
ethanol_water                                                           1349
                                                                        ... 
(s)-2-aminopropanoic acid_water                                            1
l-serine_water                                                             1
l-threonine_water                                                          1
1-butyl-3-methylimidazolium tetrafluoroborate_nitromethane                 1
1-butyl-3-methylimidazolium tetrafluoroborate_n,n-dimethylethanamide       1
Name: count, Length: 470, dtype: int64

Mean count per combination: 226.3468085106383

Mean count 